In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "HMVCL_X_view1_alpha.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "HMVCL_X_view2_stats.npy")
Y_LABELS_PATH = os.path.join(BASE_PATH, "HMVCL_y_labels.csv")
ENCODER_CNN_WEIGHTS = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN2.weights.h5")

LABEL_PERCENTAGE = 0.20  # try different Label Scarcity

# --- Helper: Generic Trainer ---
def train_task_head(X_train, y_train, X_test, y_test, task_name, use_smote=False):
    print(f"\n=== Training Task: {task_name} ===")

    # 1. Handle Class Imbalance
    if use_smote:
        print(f"   Applying SMOTE for {task_name}...")
        # k_neighbors=1 is safe for extremely rare classes
        smote = SMOTE(k_neighbors=1, random_state=42)
        try:
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
        except ValueError:
             # Fallback if a class has < 2 samples
            print("   ! SMOTE failed (too few samples). Using original data.")
            X_train_bal, y_train_bal = X_train, y_train
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # 2. Compute Weights (Double protection)
    weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # 3. Train XGBoost
    clf = xgb.XGBClassifier(
        n_estimators=150, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss' # or 'logloss' for binary
    )
    clf.fit(X_train_bal, y_train_bal, sample_weight=weights)

    # 4. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"   >>> {task_name} Weighted F1: {f1:.4f}")
    return clf, y_pred

def get_cnn_encoder_full(input_shape):
    # Definition must match exactly
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = tf.keras.layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)
    x = tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Flatten()(x)
    h = tf.keras.layers.Dense(128, activation='relu', name="representation")(x)
    z = tf.keras.layers.Dense(64, activation='relu', name="projection")(h)
    return Model(inputs, [h, z])

def main():
    # --- 1. Load & Fuse Data ---
    print("--- Loading Data ---")
    X_view1 = np.load(X_VIEW1_PATH).astype('float32')
    X_view2 = np.load(X_VIEW2_PATH).astype('float32')
    df_labels = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # Extract CNN
    cnn_full = get_cnn_encoder_full((10, 784))
    cnn_full.load_weights(ENCODER_CNN_WEIGHTS)
    cnn_extractor = Model(inputs=cnn_full.input, outputs=cnn_full.outputs[0])
    h_cnn = cnn_extractor.predict(X_view1, batch_size=128, verbose=0)

    # Final Embeddings
    X_final = np.concatenate([X_view2, h_cnn], axis=1)

    # --- 2. Data Splitting ---
    # We must use the SAME indices for all tasks to ensure fair comparison
    # We create a generic split based on indices
    indices = np.arange(len(X_final))

    # Stratify based on Application (the most detailed label)
    # This ensures rare apps (Netflix) exist in both Train and Test
    y_stratify = LabelEncoder().fit_transform(df_labels['application'])

    train_idx, test_idx = train_test_split(
        indices, train_size=LABEL_PERCENTAGE,
        random_state=42, stratify=y_stratify
    )

    X_train, X_test = X_final[train_idx], X_final[test_idx]
    print(f"Train Size: {len(X_train)}, Test Size: {len(X_test)}")

    # --- TASK 1: BINARY (VPN vs Non-VPN) ---
    if 'binary_type' in df_labels.columns:
        le_bin = LabelEncoder()
        y_bin = le_bin.fit_transform(df_labels['binary_type'])
        y_train_bin, y_test_bin = y_bin[train_idx], y_bin[test_idx]

        # Train (Binary usually doesn't need SMOTE)
        clf_bin, preds_bin = train_task_head(X_train, y_train_bin, X_test, y_test_bin, "Binary", use_smote=False)
        print(classification_report(y_test_bin, preds_bin, target_names=le_bin.classes_))
    else:
        print("Skipping Binary Task (Column 'binary_type' not found)")

    # --- TASK 2: CATEGORY (Streaming, VoIP, etc.) ---
    if 'category' in df_labels.columns:
        le_cat = LabelEncoder()
        y_cat = le_cat.fit_transform(df_labels['category'])
        y_train_cat, y_test_cat = y_cat[train_idx], y_cat[test_idx]

        # Train (Use SMOTE because Streaming category might be small in 5% split)
        clf_cat, preds_cat = train_task_head(X_train, y_train_cat, X_test, y_test_cat, "Category", use_smote=True)
        print(classification_report(y_test_cat, preds_cat, target_names=le_cat.classes_))

    # --- TASK 3: APPLICATION (Netflix, Skype, etc.) ---
    le_app = LabelEncoder()
    y_app = le_app.fit_transform(df_labels['application'])
    y_train_app, y_test_app = y_app[train_idx], y_app[test_idx]

    # Train (Definitely need SMOTE)
    clf_app, preds_app = train_task_head(X_train, y_train_app, X_test, y_test_app, "Application", use_smote=True)
    print(classification_report(y_test_app, preds_app, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Data ---
Train Size: 524, Test Size: 2099
Skipping Binary Task (Column 'binary_type' not found)

=== Training Task: Category ===
   Applying SMOTE for Category...
   ! SMOTE failed (too few samples). Using original data.
   >>> Category Weighted F1: 0.9039
               precision    recall  f1-score   support

         Chat       0.59      0.70      0.64       122
        Email       0.89      0.92      0.91       106
File Transfer       0.90      0.90      0.90       650
          P2P       0.96      0.99      0.97       289
    Streaming       0.91      0.89      0.90       207
         VoIP       0.95      0.91      0.93       725

     accuracy                           0.90      2099
    macro avg       0.86      0.88      0.87      2099
 weighted avg       0.91      0.90      0.90      2099


=== Training Task: Application ===
   Applying SMOTE for Application...
   ! SMOTE failed (too few samples). Using original data.
   >>> Application Weighted F1: 0.8599
        

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_X_view1_alpha.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_X_view2_stats.npy")
Y_LABELS_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_y_labels.csv")
ENCODER_CNN_WEIGHTS = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN2.weights.h5")

LABEL_PERCENTAGE = 0.30  # try different Label Scarcity

# --- Helper: Generic Trainer ---
def train_task_head(X_train, y_train, X_test, y_test, task_name, use_smote=False):
    print(f"\n=== Training Task: {task_name} ===")

    # 1. Handle Class Imbalance
    if use_smote:
        print(f"   Applying SMOTE for {task_name}...")
        # k_neighbors=1 is safe for extremely rare classes
        smote = SMOTE(k_neighbors=1, random_state=42)
        try:
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
        except ValueError:
             # Fallback if a class has < 2 samples
            print("   ! SMOTE failed (too few samples). Using original data.")
            X_train_bal, y_train_bal = X_train, y_train
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # 2. Compute Weights (Double protection)
    weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # 3. Train XGBoost
    clf = xgb.XGBClassifier(
        n_estimators=150, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss' # or 'logloss' for binary
    )
    clf.fit(X_train_bal, y_train_bal, sample_weight=weights)

    # 4. Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"   >>> {task_name} Weighted F1: {f1:.4f}")
    return clf, y_pred

def get_cnn_encoder_full(input_shape):
    # Definition must match exactly
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = tf.keras.layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)
    x = tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.Flatten()(x)
    h = tf.keras.layers.Dense(128, activation='relu', name="representation")(x)
    z = tf.keras.layers.Dense(64, activation='relu', name="projection")(h)
    return Model(inputs, [h, z])

def main():
    # --- 1. Load & Fuse Data ---
    print("--- Loading Data ---")
    X_view1 = np.load(X_VIEW1_PATH).astype('float32')
    X_view2 = np.load(X_VIEW2_PATH).astype('float32')
    df_labels = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # Extract CNN
    cnn_full = get_cnn_encoder_full((10, 784))
    cnn_full.load_weights(ENCODER_CNN_WEIGHTS)
    cnn_extractor = Model(inputs=cnn_full.input, outputs=cnn_full.outputs[0])
    h_cnn = cnn_extractor.predict(X_view1, batch_size=128, verbose=0)

    # Final Embeddings
    X_final = np.concatenate([X_view2, h_cnn], axis=1)

    # --- 2. Data Splitting ---
    # We must use the SAME indices for all tasks to ensure fair comparison
    # We create a generic split based on indices
    indices = np.arange(len(X_final))

    # Stratify based on Application (the most detailed label)
    # This ensures rare apps (Netflix) exist in both Train and Test
    y_stratify = LabelEncoder().fit_transform(df_labels['application'])

    train_idx, test_idx = train_test_split(
        indices, train_size=LABEL_PERCENTAGE,
        random_state=42, stratify=y_stratify
    )

    X_train, X_test = X_final[train_idx], X_final[test_idx]
    print(f"Train Size: {len(X_train)}, Test Size: {len(X_test)}")

    # --- TASK 1: BINARY (VPN vs Non-VPN) ---
    if 'binary_type' in df_labels.columns:
        le_bin = LabelEncoder()
        y_bin = le_bin.fit_transform(df_labels['binary_type'])
        y_train_bin, y_test_bin = y_bin[train_idx], y_bin[test_idx]

        # Train (Binary usually doesn't need SMOTE)
        clf_bin, preds_bin = train_task_head(X_train, y_train_bin, X_test, y_test_bin, "Binary", use_smote=False)
        print(classification_report(y_test_bin, preds_bin, target_names=le_bin.classes_))
    else:
        print("Skipping Binary Task (Column 'binary_type' not found)")

    # --- TASK 2: CATEGORY (Streaming, VoIP, etc.) ---
    if 'category' in df_labels.columns:
        le_cat = LabelEncoder()
        y_cat = le_cat.fit_transform(df_labels['category'])
        y_train_cat, y_test_cat = y_cat[train_idx], y_cat[test_idx]

        # Train (Use SMOTE because Streaming category might be small in 5% split)
        clf_cat, preds_cat = train_task_head(X_train, y_train_cat, X_test, y_test_cat, "Category", use_smote=True)
        print(classification_report(y_test_cat, preds_cat, target_names=le_cat.classes_))

    # --- TASK 3: APPLICATION (Netflix, Skype, etc.) ---
    le_app = LabelEncoder()
    y_app = le_app.fit_transform(df_labels['application'])
    y_train_app, y_test_app = y_app[train_idx], y_app[test_idx]

    # Train (Definitely need SMOTE)
    clf_app, preds_app = train_task_head(X_train, y_train_app, X_test, y_test_app, "Application", use_smote=True)
    print(classification_report(y_test_app, preds_app, target_names=le_app.classes_))

if __name__ == "__main__":
    main()

--- Loading Data ---
Train Size: 2862, Test Size: 6680

=== Training Task: Binary ===
   >>> Binary Weighted F1: 0.9656
              precision    recall  f1-score   support

      NonVPN       0.98      0.97      0.98      5350
         VPN       0.89      0.94      0.91      1330

    accuracy                           0.97      6680
   macro avg       0.94      0.95      0.95      6680
weighted avg       0.97      0.97      0.97      6680


=== Training Task: Category ===
   Applying SMOTE for Category...
   ! SMOTE failed (too few samples). Using original data.
   >>> Category Weighted F1: 0.7456
               precision    recall  f1-score   support

         Chat       0.63      0.93      0.75      1602
        Email       0.87      0.23      0.37      1275
File Transfer       0.87      0.89      0.88      1635
          P2P       0.98      0.97      0.97       253
    Streaming       0.66      0.92      0.77       206
         VoIP       0.85      0.87      0.86      1709

     